First data glance

In [10]:
import pandas as pd
import numpy as np
import pickle

# Visualization imports
import seaborn as sns
import matplotlib.pyplot as plt

# Data preparation imports
from sklearn.feature_extraction import DictVectorizer
from sklearn.pipeline import make_pipeline

# Model import
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

# Metrics import
from sklearn.metrics import root_mean_squared_error

In [11]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/MLOps-zoomcamp/02-training/mlruns/1', creation_time=1775636699181, experiment_id='1', last_update_time=1775636699181, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [12]:

def prepare_data(
    datapath,
    outcome,
    numerical_features,
    categorical_features,
    pipe=None,
    q_low=None, q_high=None,
    debug_subset=0.0,
):
    # Load data
    df = pd.read_parquet(datapath)
    df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
    df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
    df[outcome] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df[outcome] = df[outcome].dt.total_seconds() / 60
    df[categorical_features] = df[categorical_features].astype(str)

    # Training data preparation
    if pipe is None:
        q_low = df[outcome].quantile(0.02)
        q_high = df[outcome].quantile(0.98)
        df = df[(df[outcome] >= q_low) & (df[outcome] <= q_high)]
        df = df[numerical_features + categorical_features + [outcome]]
        df = df.dropna() # Remove rows with missing values before sampling
        if debug_subset:
            frac = float(debug_subset)
            df = df.sample(frac=frac, random_state=42)
        X = df[numerical_features + categorical_features]
        y = df[outcome]
        dv = DictVectorizer()
        pipe = make_pipeline(dv)
        X_train = pipe.fit_transform(X.to_dict(orient="records"))
    
    # Validation data preparation
    else:
        df = df[numerical_features + categorical_features + [outcome]]
        df = df.dropna() # Remove rows with missing values before sampling
        if debug_subset:
            frac = float(debug_subset)
            df = df.sample(frac=frac, random_state=42)
        df = df[(df[outcome] >= q_low) & (df[outcome] <= q_high)]
        X = df[numerical_features + categorical_features]
        y = df[outcome]
        X_train = pipe.transform(X.to_dict(orient="records"))

    return X_train, y, pipe, q_low, q_high

def train_model(X_train, y, y_test, model):
    model.fit(X_train, y)
    y_pred = model.predict(X_train)
    rmse = root_mean_squared_error(y, y_pred)
    print(f"RMSE: {rmse:.2f}")
    results = {
        "model": model,
        "rmse": rmse,
    }
    return results

In [13]:
NUMERICAL_FEATURES = ["trip_distance", "passenger_count", "tip_amount", "total_amount", "tolls_amount"]
CATEGORICAL_FEATURES = ['PULocationID', 'DOLocationID', "payment_type"]
OUTCOME = 'duration'
PIPE = None

train_path = '/workspaces/MLOps-zoomcamp/data/yellow_tripdata_2020-01.parquet'
test_path = '/workspaces/MLOps-zoomcamp/data/yellow_tripdata_2020-03.parquet'

X_train, y_train, pipe, q_low, q_high = prepare_data(train_path, OUTCOME, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, pipe=None, debug_subset=0.5)
X_test, y_test, _, _, _ = prepare_data(test_path, OUTCOME, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, pipe, q_low, q_high, debug_subset=0.5)

In [14]:
model = LinearRegression()
#model = RandomForestRegressor(n_estimators=100, random_state=42)
#model = Lasso(alpha = 0.01)
result = train_model(X_train, y_train, y_test, model)

# Save the results to registry as pickle file
with open('../registry/model.pkl', 'wb') as f_out:
    pickle.dump(result, f_out)

RMSE: 4.04


In [15]:
with mlflow.start_run():
    mlflow.set_tag("developer", "johan")

    mlflow.log_param("train-data-path", train_path)
    mlflow.log_param("valid-data-path", test_path)

    alpha = 0.01
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse", rmse)
   

In [16]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
import optuna
import mlflow

In [25]:

# X_t: X_train with no data leakage.
# X_v: X_val with no data leakage for later test.
X_t, X_v, y_t, y_v = train_test_split(
    X_train, y_train, 
    test_size=0.2, 
    random_state=42)

def make_objective(parent_run_id):
    def objective(trial):
        params = {
            "max_depth":trial.suggest_int("max_depth", 2, 5),
            "learning_rate":trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "eval_metric": "rmse",  # or "mae", "logloss" etc depending on your task
            "objective": "reg:squarederror",  # add this too if regression
        }


        dtrain = xgb.DMatrix(X_t, label=y_t)
        dval = xgb.DMatrix(X_v, label=y_v)

        evals_results = {}
        model = xgb.train(
            params, 
            dtrain,
            num_boost_round=1000,
            evals=[(dval, "validation")],
            early_stopping_rounds=50,
            evals_result=evals_results,
            verbose_eval=False,
        )
        
        print("best_score:", model.best_score)
        print("best_iteration:", model.best_iteration)
        print("evals_result keys:", evals_results.keys())
        print("evals_result:", evals_results)
        
        eval_rmse = evals_results["validation"]["rmse"]


        best_score = eval_rmse[np.argmin(eval_rmse)]
        best_round = np.argmin(eval_rmse) + 1
        
        print(best_score)
        print(best_round)

        dtest = xgb.DMatrix(X_test, label=y_test)

        y_pred = model.predict(dtest)
        rmse = root_mean_squared_error(y_pred, y_test)
        
        print(rmse)


        with mlflow.start_run(nested=True, parent_run_id=parent_run_id):
            mlflow.log_params(params)
            mlflow.log_metric("val_rmse", best_score)
            mlflow.log_metric("test_rmse", rmse)
            mlflow.xgboost.log_model(model, name="model")  # ← logs to this specific run
            
        return best_score
    return objective

In [26]:
with mlflow.start_run(run_name="optuna_study") as parent:
    study = optuna.create_study(direction="minimize")
    study.optimize(make_objective(parent.info.run_id), n_trials=10)

[I 2026-04-15 06:18:57,505] A new study created in memory with name: no-name-1a953964-dbcd-443d-9abb-b1f40b9874a0


2026/04/15 06:20:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


best_score: 2.2028742198153752
best_iteration: 999
evals_result keys: dict_keys(['validation'])
evals_result: {'validation': OrderedDict({'rmse': [7.975640107726355, 7.878242816318822, 7.782346715381679, 7.688213205379373, 7.595691995938792, 7.504519121986128, 7.414642128946582, 7.326576405000661, 7.23967903532809, 7.154531872756906, 7.070801752373945, 6.988284132688535, 6.907423170704784, 6.827613746995957, 6.749602270893895, 6.672584953164196, 6.596946783400485, 6.522622943726587, 6.4497094913072175, 6.377905616156506, 6.307641350233035, 6.23832720212604, 6.170278995751278, 6.103423331399022, 6.037727223879495, 5.9734152244313625, 5.910043420728278, 5.847741390551672, 5.786833918867872, 5.726276899902795, 5.667292255531444, 5.6193218886114575, 5.5624653050477715, 5.505877767658798, 5.460707490622219, 5.4068815238558185, 5.353903837579774, 5.302142006357812, 5.2509127513364575, 5.2008744536770175, 5.150976909126986, 5.102666380815279, 5.055221271180739, 5.008852455571863, 4.9632053504

[I 2026-04-15 06:21:09,052] Trial 0 finished with value: 2.2028742198153752 and parameters: {'max_depth': 4, 'learning_rate': 0.01471227911668616, 'subsample': 0.8701941812038985, 'colsample_bytree': 0.97145458898034, 'min_child_weight': 3}. Best is trial 0 with value: 2.2028742198153752.
2026/04/15 06:23:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 06:24:03 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 013df815a4634abb9b8b2383a3b46d84. Failed operations: [MlflowException("Changing param values is not allowed. Params were already logged=\'[{\'key\': \'max_depth\', \'old_value\': \'4\', \'new_value\': \'5\'}, {\'key\': \'learning_rate\', \'old_value\': \'0.01471227911668616\', \'new_value\': \'0.0011360976628377335\'}, {\'key

best_score: 3.752702932274012
best_iteration: 999
evals_result keys: dict_keys(['validation'])
evals_result: {'validation': OrderedDict({'rmse': [8.066860981598483, 8.059015005361118, 8.051177680074712, 8.043355063710372, 8.035535705556489, 8.027827292626345, 8.020032410613421, 8.012246228631742, 8.004553717513941, 7.996788769522767, 7.9890468375707835, 7.98130272087066, 7.973564684600255, 7.9658468282761605, 7.958130836789375, 7.9504425599306305, 7.9427433655944535, 7.935069231644304, 7.927388561934796, 7.920784452002204, 7.913129691432148, 7.905491455383655, 7.8978500130810945, 7.890226186019202, 7.882610377950357, 7.875010011759511, 7.867412562002884, 7.859827458429164, 7.852247665582056, 7.844675227318361, 7.837115965301322, 7.830610739327437, 7.823079478347811, 7.815546864869025, 7.809068695668083, 7.801635063207547, 7.794134963263114, 7.787692158652146, 7.780216767353236, 7.772751442326379, 7.76528735858385, 7.757841801343656, 7.7504060454313315, 7.746118747923779, 7.738703201439

...

[I 2026-04-15 06:38:44,546] Trial 7 finished with value: 5.949380186465851 and parameters: {'max_depth': 5, 'learning_rate': 0.00038480373332696005, 'subsample': 0.9579975711193147, 'colsample_bytree': 0.9152180396880285, 'min_child_weight': 8}. Best is trial 6 with value: 2.024846857109085.
2026/04/15 06:40:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 06:40:10 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 013df815a4634abb9b8b2383a3b46d84. Failed operations: [MlflowException("Changing param values is not allowed. Params were already logged=\'[{\'key\': \'max_depth\', \'old_value\': \'4\', \'new_value\': \'2\'}, {\'key\': \'learning_rate\', \'old_value\': \'0.01471227911668616\', \'new_value\': \'0.0036435222253470637\'}, {\'

best_score: 3.2079860792243062
best_iteration: 999
evals_result keys: dict_keys(['validation'])
evals_result: {'validation': OrderedDict({'rmse': [8.052311642294365, 8.030029116653614, 8.007832568427332, 7.985739876434533, 7.96374464829582, 7.941852903967956, 7.920051375152551, 7.898354678300017, 7.876751051980654, 7.855226250375333, 7.833810716674842, 7.812462757750264, 7.791225622955882, 7.77008823230795, 7.749049189420465, 7.728084231213512, 7.707192614455829, 7.686403979083989, 7.665683858870837, 7.645072598150311, 7.624539274546486, 7.604104316746697, 7.583742922984921, 7.563489027170763, 7.543301674487026, 7.52318158126045, 7.503204796870579, 7.483278375573233, 7.4634174687020405, 7.443665544420182, 7.423968468520301, 7.4043668157278635, 7.384867657248704, 7.365473265010618, 7.346138691618891, 7.326872459409024, 7.307714093165202, 7.288616892311472, 7.269615029378376, 7.250681312792488, 7.231854301843609, 7.2130838347464445, 7.194437799677022, 7.1758458474661, 7.1573245184543985,

[I 2026-04-15 06:40:21,010] Trial 8 finished with value: 3.2079860792243062 and parameters: {'max_depth': 2, 'learning_rate': 0.0036435222253470637, 'subsample': 0.9301819564407303, 'colsample_bytree': 0.9937963348436496, 'min_child_weight': 9}. Best is trial 6 with value: 2.024846857109085.
2026/04/15 06:42:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 06:42:55 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during xgboost autologging: The following failures occurred while performing one or more logging operations: [MlflowException('Failed to perform one or more operations on the run with ID 013df815a4634abb9b8b2383a3b46d84. Failed operations: [MlflowException("Changing param values is not allowed. Params were already logged=\'[{\'key\': \'max_depth\', \'old_value\': \'4\', \'new_value\': \'5\'}, {\'key\': \'learning_rate\', \'old_value\': \'0.01471227911668616\', \'new_value\': \'0.003957226566886268\'}, {\'k

best_score: 2.422215703203123
best_iteration: 999
evals_result keys: dict_keys(['validation'])
evals_result: {'validation': OrderedDict({'rmse': [8.047386785652899, 8.020154567670154, 7.993056789727045, 7.969797994586427, 7.94323688228721, 7.928840676507487, 7.9021453046785055, 7.875569774175008, 7.849427680783842, 7.823071345722608, 7.796900682671566, 7.77088475511939, 7.744988853413115, 7.719385723220938, 7.693643142478125, 7.668071126588026, 7.642558558349699, 7.61718635735307, 7.5921879827478165, 7.570555929820713, 7.5455593865363, 7.524176559013829, 7.499612777311344, 7.4747871696494865, 7.450055811166406, 7.4255546780125075, 7.401184691706139, 7.380425096850836, 7.356137087111773, 7.3355266137111235, 7.311430023192646, 7.291016064923665, 7.2671475724035135, 7.243491328861157, 7.223370198100888, 7.2001837186512905, 7.176717830702234, 7.156913337135469, 7.133908271709208, 7.110748066741651, 7.087852145190863, 7.065017211488991, 7.042169335439831, 7.029354304925183, 7.00676850257919

[I 2026-04-15 06:43:19,783] Trial 9 finished with value: 2.422215703203123 and parameters: {'max_depth': 5, 'learning_rate': 0.003957226566886268, 'subsample': 0.8077945828411003, 'colsample_bytree': 0.8589775741273342, 'min_child_weight': 8}. Best is trial 6 with value: 2.024846857109085.
